<a href="https://colab.research.google.com/github/mcuartero/fine-grained-visual-recognition/blob/main/02-animal-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import os
import copy
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

import torch
import timm
import torchvision.transforms as transforms

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score,
    RandomizedSearchCV, train_test_split
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import VotingClassifier
from scipy.stats import loguniform
from xgboost import XGBClassifier

# GPU-accelerated models (cuML) with sklearn fallback
try:
    from cuml.linear_model import LogisticRegression
    from cuml.svm import SVC
    print("Using cuML (GPU) for Logistic Regression and SVM")
except ImportError:
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    print("cuML not found, falling back to sklearn (CPU)")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch device: {DEVICE}")

cuML not found, falling back to sklearn (CPU)
Torch device: cpu


In [19]:
import sys
from pathlib import Path

# 1. Check if we are running inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    print("☁️ Running on Google Colab. Setting up Drive paths...")
    
    # 2. Mount Google Drive
    drive.mount('/content/drive')

    # 3. Define paths
    ZIP_PATH = Path("/content/drive/MyDrive/animals.zip")
    EXTRACT_DIR = Path("/content/data")

    checkpoint_1 = EXTRACT_DIR / "train_metadata.csv"
    checkpoint_2 = EXTRACT_DIR / "animals" / "train_metadata.csv"

    # 4. Run the unzip logic
    if checkpoint_1.exists() or checkpoint_2.exists():
        print("Data already unzipped.")
    else:
        if ZIP_PATH.exists():
            print("Unzipping now...")
            !unzip -q {ZIP_PATH} -d {EXTRACT_DIR}
            print("Unzipping complete!")
        else:
            print("Could not find 'animals.zip' in the root of your Google Drive.")
else:
    print("Running locally. Skipping Google Drive & unzip setup.")

Running locally. Skipping Google Drive & unzip setup.


In [20]:
# Configuration

"""
LOCAL (comment/uncomment when not using)
"""
BASE_DIR = Path("data/animals")

TRAIN_META = BASE_DIR / "train_metadata.csv"
TEST_META = BASE_DIR / "test_metadata.csv"
COLOR_HIST = BASE_DIR / "color_histogram.csv"
HOG_PCA = BASE_DIR / "hog_pca.csv"
ADDITIONAL_FEATS = BASE_DIR / "additional_features.csv"

TRAIN_VIT_CACHE = BASE_DIR / "train_vit_base_embeddings.npy"
TEST_VIT_CACHE  = BASE_DIR / "test_vit_base_embeddings.npy"
TRAIN_RN_CACHE  = BASE_DIR / "train_resnet50_embeddings.npy"
TEST_RN_CACHE   = BASE_DIR / "test_resnet50_embeddings.npy"

SUBMISSION_PATH = BASE_DIR / "submission.csv"

"""
GOOGLE COLAB PATHS (comment/uncomment when not using)
"""

# BASE_DIR = Path("/content/data/")

# TRAIN_META = BASE_DIR / "train_metadata.csv"
# TEST_META = BASE_DIR / "test_metadata.csv"
# COLOR_HIST = BASE_DIR / "color_histogram.csv"
# HOG_PCA = BASE_DIR / "hog_pca.csv"
# ADDITIONAL_FEATS = BASE_DIR / "additional_features.csv"

# TRAIN_VIT_CACHE = BASE_DIR / "train_vit_base_embeddings.npy"
# TEST_VIT_CACHE  = BASE_DIR / "test_vit_base_embeddings.npy"
# TRAIN_RN_CACHE   = BASE_DIR / "train_resnet50_embeddings.npy"
# TEST_RN_CACHE    = BASE_DIR / "test_resnet50_embeddings.npy"

# SUBMISSION_PATH = BASE_DIR / "submission.csv"

"""
HARDWARE SETTINGS
"""

BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


In [21]:
# Load CSV Features

print("\n--- Loading CSV features ---")

train_meta = pd.read_csv(TRAIN_META)
test_meta  = pd.read_csv(TEST_META)

color = pd.read_csv(COLOR_HIST)
hog = pd.read_csv(HOG_PCA)
add = pd.read_csv(ADDITIONAL_FEATS)

def merge_features(meta, *feature_dfs):
    df = meta.copy()
    for fdf in feature_dfs:
        df = df.merge(fdf, on="image_id", how="left")
    return df

train_df = merge_features(train_meta, color, hog, add)
test_df  = merge_features(test_meta,  color, hog, add)

meta_cols = ["image_id", "image_path", "class_id", "class_name"]
csv_feature_cols = [c for c in train_df.columns if c not in meta_cols]

print(f"Train samples: {len(train_df)},  Test samples: {len(test_df)}")
print(f"CSV feature dims: {len(csv_feature_cols)}")


--- Loading CSV features ---
Train samples: 3750,  Test samples: 1250
CSV feature dims: 219


In [22]:
def extract_embeddings(image_paths, base_dir, cache_path, model_name, batch_size=32):
    """
    Extract embeddings from any timm model.
    Uses a disk cache so re-running the script is fast.
    """
    if cache_path.exists():
        print(f"  Loading cached embeddings from {cache_path}")
        return np.load(cache_path)

    print(f"  Extracting {model_name} embeddings for {len(image_paths)} images ...")

    model = timm.create_model(model_name, pretrained=True, num_classes=0)
    model.eval().to(DEVICE)

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    all_embeddings = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i : i + batch_size]
        imgs = []
        for rel_path in batch_paths:
            full_path = base_dir / rel_path
            try:
                img = Image.open(full_path).convert("RGB")
            except Exception:
                img = Image.new("RGB", (64, 64), (128, 128, 128))
            imgs.append(preprocess(img))

        batch_tensor = torch.stack(imgs).to(DEVICE)
        with torch.no_grad():
            emb = model(batch_tensor).cpu().numpy()
        all_embeddings.append(emb)

        if (i // batch_size) % 10 == 0:
            print(f"    {min(i + batch_size, len(image_paths))}/{len(image_paths)}")

    embeddings = np.vstack(all_embeddings)
    np.save(cache_path, embeddings)
    print(f"  Saved to {cache_path}")
    return embeddings


# ── ViT-Base/16: 768-dim, richer than ViT-Small (384-dim) ──────────────────
print("\n--- Extracting ViT-Base embeddings ---")
train_vit = extract_embeddings(
    train_df["image_path"].values, BASE_DIR,
    TRAIN_VIT_CACHE, "vit_base_patch16_224", BATCH_SIZE
)
test_vit = extract_embeddings(
    test_df["image_path"].values, BASE_DIR,
    TEST_VIT_CACHE, "vit_base_patch16_224", BATCH_SIZE
)

# ── ResNet-50: 2048-dim, CNN complements ViT's attention-based features ──────
print("\n--- Extracting ResNet-50 embeddings ---")
train_rn = extract_embeddings(
    train_df["image_path"].values, BASE_DIR,
    TRAIN_RN_CACHE, "resnet50", BATCH_SIZE
)
test_rn = extract_embeddings(
    test_df["image_path"].values, BASE_DIR,
    TEST_RN_CACHE, "resnet50", BATCH_SIZE
)

print(f"ViT-Base shape:  {train_vit.shape}")
print(f"ResNet-50 shape: {train_rn.shape}")


--- Extracting ViT-Base embeddings ---
  Loading cached embeddings from data\animals\train_vit_base_embeddings.npy
  Loading cached embeddings from data\animals\test_vit_base_embeddings.npy

--- Extracting ResNet-50 embeddings ---
  Loading cached embeddings from data\animals\train_resnet50_embeddings.npy
  Loading cached embeddings from data\animals\test_resnet50_embeddings.npy
ViT-Base shape:  (3750, 768)
ResNet-50 shape: (3750, 2048)


In [23]:
# Combine and normalise features

print("\n--- Combining and normalising features ---")

X_train_csv = np.nan_to_num(train_df[csv_feature_cols].values.astype(np.float32))
X_test_csv  = np.nan_to_num(test_df[csv_feature_cols].values.astype(np.float32))

scaler_csv = StandardScaler()
X_train_csv = scaler_csv.fit_transform(X_train_csv)
X_test_csv  = scaler_csv.transform(X_test_csv)

scaler_vit = StandardScaler()
X_train_vit = scaler_vit.fit_transform(train_vit)
X_test_vit  = scaler_vit.transform(test_vit)

scaler_rn = StandardScaler()
X_train_rn = scaler_rn.fit_transform(train_rn)
X_test_rn  = scaler_rn.transform(test_rn)

# Concatenate: [ViT-Base | ResNet-50 | CSV features]
X_train = np.hstack([X_train_vit, X_train_rn, X_train_csv])
X_test  = np.hstack([X_test_vit,  X_test_rn,  X_test_csv])

y_train = train_df["class_id"].values

print(f"Total feature dims: {X_train.shape[1]}")
print(f"  ViT-Base:  {X_train_vit.shape[1]}")
print(f"  ResNet-50: {X_train_rn.shape[1]}")
print(f"  CSV:       {X_train_csv.shape[1]}")


--- Combining and normalising features ---
Total feature dims: 3035
  ViT-Base:  768
  ResNet-50: 2048
  CSV:       219


In [24]:
# Cross validation helper

def cv_evaluate(model, X, y, n_splits=5, label="Model"):
    """Stratified 5-fold cross-validation."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
    print(f"  {label}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {np.round(scores, 4)})")
    return scores.mean()


In [25]:
print("\n--- Model 1: Logistic Regression ---")

# Check if LogisticRegression is from cuML (GPU) or scikit-learn (CPU)
is_cuml = lr.__module__.startswith("cuml")

if is_cuml:
    print("Initializing GPU-accelerated Logistic Regression (cuML)")
    solver_name = "qn"
else:
    print("Initializing CPU Logistic Regression (scikit-learn)")
    solver_name = "lbfgs"

lr = LogisticRegression(
    C=10.0,
    max_iter=2000,
    solver=solver_name,
    class_weight="balanced"
)

lr_cv = cv_evaluate(lr, X_train, y_train, label="Logistic Regression")


--- Model 1: Logistic Regression ---
Initializing CPU Logistic Regression (scikit-learn)
  Logistic Regression: 0.9480 ± 0.0102  (folds: [0.9293 0.9547 0.9533 0.9573 0.9453])


In [26]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

print("\n--- Model 2: SVM (RBF) with RandomizedSearchCV ---")

# 1. Detect if we are using cuML (GPU) or scikit-learn (CPU)
is_cuml = SVC.__module__.startswith("cuml")

# 2. Adjust settings dynamically based on hardware to save time on CPU
if is_cuml:
    print("Running on GPU (cuML). Maximizing search parameters.")
    svc_kwargs = {"kernel": "rbf", "probability": True} # cuML doesn't take random_state
    search_iterations = 15
    search_verbose = 3 # Show comprehensive progress updates on GPU
else:
    print("Running on CPU (scikit-learn)...")
    svc_kwargs = {"kernel": "rbf", "probability": False, "random_state": 42} # Disable prob temporarily for speed
    search_iterations = 5 # Reduce iterations on CPU so it completes in minutes, not hours
    search_verbose = 3  # High verbosity so you can see each fit complete live

param_dist = {
    "C": loguniform(1, 1000),         # Search log-uniformly in [1, 1000]
    "gamma": loguniform(1e-4, 1e-1),   # Search log-uniformly in [0.0001, 0.1]
}

# 3. Initialize and fit the search
svm_search = RandomizedSearchCV(
    SVC(**svc_kwargs),
    param_distributions=param_dist,
    n_iter=search_iterations,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=search_verbose
)

print(f"Starting RandomizedSearchCV ({search_iterations} iterations, 3 folds)...")
svm_search.fit(X_train, y_train)

print(f"  Best params found: {svm_search.best_params_}")
print(f"  Best CV score during search: {svm_search.best_score_:.4f}")

# 4. Final Best Estimator Handling
if not is_cuml:
    # On CPU, we must rebuild the best model with probability=True 
    # so it can be safely used later inside your Soft Voting Ensemble!
    print("Refitting best CPU estimator to enable probabilities for the Ensemble...")
    svm = SVC(
        kernel="rbf", 
        probability=True, 
        random_state=42, 
        **svm_search.best_params_
    )
    svm.fit(X_train, y_train)
else:
    # On GPU, probability=True was already on, so we just grab it directly
    svm = svm_search.best_estimator_

svm_cv = svm_search.best_score_


--- Model 2: SVM (RBF) with RandomizedSearchCV ---
Running on CPU (scikit-learn)...
Starting RandomizedSearchCV (5 iterations, 3 folds)...
Fitting 3 folds for each of 5 candidates, totalling 15 fits
  Best params found: {'C': np.float64(2.9380279387035344), 'gamma': np.float64(0.00029375384576328325)}
  Best CV score during search: 0.9395
Refitting best CPU estimator to enable probabilities for the Ensemble...


In [27]:
from xgboost import XGBClassifier

print("\n--- Model 3: XGBoost ---")

# Adjust the number of trees based on the device you set early on
if DEVICE == "cuda":
    print("Running XGBoost on GPU (CUDA)")
    num_trees = 500
else:
    print("Running XGBoost on CPU (Downscaling trees to save your local processing time)")
    num_trees = 100

xgb = XGBClassifier(
    n_estimators=num_trees,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.6,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    device=DEVICE
)

xgb_cv = cv_evaluate(xgb, X_train, y_train, label="XGBoost")


--- Model 3: XGBoost ---
Running XGBoost on CPU (Downscaling trees to save your local processing time)
  XGBoost: 0.9355 ± 0.0135  (folds: [0.916  0.956  0.9413 0.9373 0.9267])


In [28]:
print("\n--- Model 4: Soft Voting Ensemble ---")

# 1. Grab tuned SVM hyperparameters from your previous search cell
best_C = svm_search.best_params_["C"]
best_gamma = svm_search.best_params_["gamma"]

# 2. Check if we are using cuML (GPU) or scikit-learn (CPU) for the linear models
is_cuml = LogisticRegression.__module__.startswith("cuml")

# 3. Handle specific architecture dependencies dynamically
if is_cuml or DEVICE == "cuda":
    print("Configuring Ensemble for GPU Execution (cuML + CUDA XGBoost)")
    lr_inst = LogisticRegression(C=10.0, max_iter=2000, solver="qn", class_weight="balanced")
    svm_inst = SVC(kernel="rbf", probability=True, C=best_C, gamma=best_gamma) # cuML drops random_state
    xgb_trees = 500
else:
    print("Configuring Ensemble for CPU Execution (scikit-learn + standard XGBoost)")
    lr_inst = LogisticRegression(C=10.0, max_iter=2000, solver="lbfgs", class_weight="balanced")
    svm_inst = SVC(kernel="rbf", probability=True, C=best_C, gamma=best_gamma, random_state=42)
    xgb_trees = 100 # Trimmed from 500 for local sanity check

# 4. Assemble the Voting Classifier
ensemble = VotingClassifier(
    estimators=[
        ("lr", lr_inst),
        ("svm", svm_inst),
        ("xgb", XGBClassifier(
                    n_estimators=xgb_trees, 
                    max_depth=5, 
                    learning_rate=0.05,
                    subsample=0.8, 
                    colsample_bytree=0.6,
                    min_child_weight=3, 
                    gamma=0.1,
                    reg_alpha=0.1, 
                    reg_lambda=1.0,
                    eval_metric="mlogloss",
                    random_state=42, 
                    n_jobs=-1, 
                    device=DEVICE)),
    ],
    voting="soft"   # Averages predicted probabilities
)

# 5. Drop straight into your existing global cv_evaluate execution!
ensemble_cv = cv_evaluate(ensemble, X_train, y_train, label="Soft Voting Ensemble")


--- Model 4: Soft Voting Ensemble ---
Configuring Ensemble for CPU Execution (scikit-learn + standard XGBoost)
  Soft Voting Ensemble: 0.9533 ± 0.0108  (folds: [0.9347 0.968  0.952  0.9573 0.9547])


In [29]:
print("\n--- Cross-validation summary ---")
results = {
    "Logistic Regression":  lr_cv,
    "SVM RBF (tuned)":      svm_cv,
    "XGBoost":              xgb_cv,
    "Soft Voting Ensemble": ensemble_cv,
}
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name}: {score:.4f}")

best_name  = max(results, key=results.get)
model_map  = {
    "Logistic Regression":  lr,
    "SVM RBF (tuned)":      svm,
    "XGBoost":              xgb,
    "Soft Voting Ensemble": ensemble,
}
best_model = model_map[best_name]
print(f"\nBest model: {best_name} ({results[best_name]:.4f})")

print(f"Fitting {best_name} on full training set ...")
best_model.fit(X_train, y_train)
train_preds = best_model.predict(X_train)
print(f"Training accuracy (sanity check): {accuracy_score(y_train, train_preds):.4f}")



--- Cross-validation summary ---
  Soft Voting Ensemble: 0.9533
  Logistic Regression: 0.9480
  SVM RBF (tuned): 0.9395
  XGBoost: 0.9355

Best model: Soft Voting Ensemble (0.9533)
Fitting Soft Voting Ensemble on full training set ...
Training accuracy (sanity check): 1.0000


In [ ]:

print("\n--- Generating confusion matrix on 20% holdout ---")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

eval_model = copy.deepcopy(best_model)
eval_model.fit(X_tr, y_tr)
val_preds = eval_model.predict(X_val)
val_acc   = accuracy_score(y_val, val_preds)
print(f"Holdout validation accuracy: {val_acc:.4f}")

class_names = (
    train_df[["class_id", "class_name"]]
    .drop_duplicates()
    .sort_values("class_id")["class_name"]
    .tolist()
)

cm = confusion_matrix(y_val, val_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix — {best_name} (val acc: {val_acc:.3f})")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()

output_dir = "figures"
output_dir.mkdir(parents=True, exist_ok=True)
safe_model_name = best_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
plt.savefig(output_dir / f"{safe_model_name}.png", dpi=150)
print(f"Saved: {output_dir / f'{safe_model_name}.png'}")

print("\nClassification Report:")
print(classification_report(y_val, val_preds, target_names=class_names))


--- Generating confusion matrix on 20% holdout ---
Holdout validation accuracy: 0.9373
Saved: data\animals\figures\soft_voting_ensemble.png

Classification Report:
              precision    recall  f1-score   support

        bird       0.97      0.93      0.95        75
   butterfly       0.99      0.93      0.96        75
         cat       0.81      0.85      0.83        75
        deer       0.89      0.95      0.92        75
         dog       0.91      0.81      0.86        75
    elephant       0.97      0.97      0.97        75
        frog       0.94      0.99      0.96        75
       horse       0.96      0.96      0.96        75
       sheep       1.00      0.99      0.99        75
      spider       0.95      0.99      0.97        75

    accuracy                           0.94       750
   macro avg       0.94      0.94      0.94       750
weighted avg       0.94      0.94      0.94       750

